In [1]:
!pip install langchain langchain_groq mysql-connector-python requests python-dotenv

   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   -- ------------------------------------- 1.0/16.5 MB 4.3 MB/s eta 0:00:04
   ----- ---------------------------------- 2.4/16.5 MB 5.0 MB/s eta 0:00:03
   -------- ------------------------------- 3.4/16.5 MB 5.1 MB/s eta 0:00:03
   ------------ --------------------------- 5.0/16.5 MB 5.7 MB/s eta 0:00:03
   ---------------- ----------------------- 6.8/16.5 MB 6.2 MB/s eta 0:00:02
   ---------------------- ----------------- 9.4/16.5 MB 7.1 MB/s eta 0:00:01
   --------------------------- ------------ 11.5/16.5 MB 7.5 MB/s eta 0:00:01
   ------------------------------------ --- 15.2/16.5 MB 8.6 MB/s eta 0:00:01
   ---------------------------------------- 16.5/16.5 MB 8.6 MB/s  0:00:02


In [2]:
import os
import requests
import mysql.connector
from dotenv import load_dotenv
from langchain_groq import ChatGroq

In [6]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
NEWS_API_KEY = os.getenv("NEWS_API_KEY")

print("Groq Key:", GROQ_API_KEY)
print("News Key:", NEWS_API_KEY)

Groq Key: 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
News Key: 8d1c86fec55d4d658c06bfb1189778fe


In [7]:
db = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = db.cursor()

print("Database Connected ✅")

Database Connected ✅


In [8]:
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant"
)

print("LLM Ready ✅")

LLM Ready ✅


In [9]:
def save_user(name, email, interest):
    query = "INSERT INTO users (name, email, interest) VALUES (%s, %s, %s)"
    cursor.execute(query, (name, email, interest))
    db.commit()

In [10]:
def fetch_news(interest):
    url = f"https://newsapi.org/v2/everything?q={interest}&apiKey={NEWS_API_KEY}"
    response = requests.get(url).json()
    return response["articles"][:5]

In [11]:
def summarize_news(article):
    text = article["title"] + " " + str(article["description"])

    prompt = f"""
    You are helping students preparing for exams.

    Summarize this news in:
    - Simple language
    - Key points
    - Why important for exams

    News:
    {text}
    """

    response = llm.invoke(prompt)
    return response.content

In [12]:
print("===== AI NEWS READER =====")

name = input("Enter your name: ")
email = input("Enter your email: ")
interest = input("Enter your interest (business, tech, economy): ")

save_user(name, email, interest)

print("\nFetching news...\n")

articles = fetch_news(interest)

for i, article in enumerate(articles):
    print(f"\n📰 News {i+1}: {article['title']}\n")
    
    summary = summarize_news(article)
    print(summary)
    
    print("-" * 50)

===== AI NEWS READER =====


Enter your name:  SHWET
Enter your email:  SHWETYADAV@149GMAIL.COM
Enter your interest (business, tech, economy):  BUSINESS



Fetching news...


📰 News 1: Can Puck reinvent the news business for the influencer age?

**Simple Language:**
Puck is a new media company that creates newsletters with famous people. They charge people to read these newsletters, and it's all part of a package deal. This way, people can get interesting news from someone they like.

**Key Points:**

1. Puck is a media company that creates newsletters with famous people.
2. These newsletters are part of a subscription package, not free.
3. People pay to read these newsletters because they like the person writing it.

**Why important for exams:**
Understanding how media companies like Puck work is important for exams because it shows how the way we get news is changing. In the past, people mostly got news from newspapers or TV, but now they get it from social media, blogs, and newsletters. This is a good example of how technology and social media are changing the way we consume information. For exams, you might be asked questions about h

In [13]:
print("===== AI NEWS READER =====")

while True:
    print("\n--- New Search ---")

    name = input("Enter your name: ")
    email = input("Enter your email: ")
    interest = input("Enter your interest (business, tech, economy): ")

    save_user(name, email, interest)

    print("\nFetching news...\n")

    articles = fetch_news(interest)

    for i, article in enumerate(articles):
        print(f"\n📰 News {i+1}: {article['title']}\n")

        summary = summarize_news(article)
        print(summary)

        print("-" * 50)

    # 🔴 Ask user to continue or exit
    choice = input("\nDo you want to search again? (yes/no): ")

    if choice.lower() != "yes":
        print("Exiting... Thank you!")
        break

===== AI NEWS READER =====

--- New Search ---


Enter your name:  AMIT
Enter your email:  SSYADAUVANSHI150@GMAIL.COM
Enter your interest (business, tech, economy):  BUSINESS



Fetching news...


📰 News 1: Can Puck reinvent the news business for the influencer age?

**Simple Language:**

Puck is a new media company that creates newsletters with famous people. These newsletters are part of a paid subscription package. The company wants to change the way news is shared, especially for people who follow influencers.

**Key Points:**

1. Puck is a media company that creates newsletters with famous people.
2. These newsletters are part of a paid subscription package.
3. The company is trying to change the way news is shared.

**Why Important for Exams:**

This news is important because it shows how media and news are changing in the world. Understanding how media companies like Puck are using influencers and newsletters can help you in exams that cover media studies, business, or communication. You might be asked questions about how media companies are adapting to new technologies and changing consumer habits.
--------------------------------------------------

📰


Do you want to search again? (yes/no):  YES



--- New Search ---


Enter your name:  rahul
Enter your email:  shwetyadav149@gmail.com
Enter your interest (business, tech, economy):  sports



Fetching news...


📰 News 1: Fubo’s mobile app is pushing more sports highlights

**Simple Language:**

A sports app called Fubo is getting a new update. This update will include short videos about your favorite sports teams and live videos when you open the app. You can think of it like a sports news feed with the latest updates and action.

**Key Points:**

1. Fubo is updating its mobile app.
2. The update includes short-form videos with sports news and highlights.
3. A live video carousel will be available when you open the app.

**Why Important for Exams:**

While this news may not directly relate to exam subjects like math, science, or history, it's essential for students who enjoy sports and want to stay updated on the latest news. For exams, this news can be useful in a few ways:

- Critical thinking and analysis: Students can practice analyzing the information and making connections between different sports news and events.
- Communication skills: Students can write or discuss


Do you want to search again? (yes/no):  no


Exiting... Thank you!


In [14]:
while True:
    print("\n===== AI NEWS READER =====")

    name = input("Enter your name: ")
    email = input("Enter your email: ")

    language = input("Enter language (english/hindi): ").lower()
    interest = input("Enter your interest (business, tech, economy): ")

    save_user(name, email, interest)

    print("\nFetching news...\n")

    # language filter
    if language == "hindi":
        url = f"https://newsapi.org/v2/everything?q={interest}&language=hi&apiKey={NEWS_API_KEY}"
    else:
        url = f"https://newsapi.org/v2/everything?q={interest}&language=en&apiKey={NEWS_API_KEY}"

    response = requests.get(url).json()
    articles = response.get("articles", [])[:5]

    for i, article in enumerate(articles):
        print(f"\n📰 News {i+1}: {article['title']}\n")

        summary = summarize_news(article)
        print(summary)

        print("-" * 50)

    choice = input("\nSearch again? (yes/no): ").lower()

    if choice == "no":
        print("Program closed. Run again anytime to read news 👍")
        break


===== AI NEWS READER =====


Enter your name:  sumit
Enter your email:  shwet123@gmail.com
Enter language (english/hindi):  hindi
Enter your interest (business, tech, economy):  economy



Fetching news...


📰 News 1: तो क्या खत्म हो जाएगा खर्ग आइलैंड? ट्रंप की ईरान को वॉर्निंग, बोले- ऐसा करना...

**Simple Language:**

अमेरिका के राष्ट्रपति डोनाल्ड ट्रंप ने ईरान को चेतावनी दी है. उन्होंने कहा है कि अगर ईरान खार्ग द्वीप पर तेल पाइपलाइनों पर हमला करता है, तो यह बहुत बड़ा नुकसान होगा.

**Key Points:**

- डोनाल्ड ट्रंप ने ईरान को चेतावनी दी है
- ईरान खार्ग द्वीप पर तेल पाइपलाइनों पर हमला करने की तैयारी कर रहा है
- यदि ईरान हमला करता है, तो यह बहुत बड़ा नुकसान होगा
- खार्ग द्वीप ईरान की अर्थव्यवस्था की रीढ़ मानी जाती है

**Why Important for Exams:**

यह खबर महत्वपूर्ण है क्योंकि इसमें अंतर्राष्ट्रीय राजनीति और अर्थव्यवस्था के बारे में जानकारी है. आप इसे पढ़कर जानकारी प्राप्त कर सकते हैं कि दुनिया के देशों के बीच क्या होता है, और इसके प्रभाव क्या हो सकते हैं.

आपके लिए यह जानना महत्वपूर्ण है कि:

- अंतर्राष्ट्रीय संबंधों में देशों के बीच कैसे समझौते और विवाद होते हैं
- दुनिया के देशों की अर्थव्यवस्था कैसे काम करती है और इसके महत्वपूर्ण हिस्से क्या हैं
- अंतर्राष्ट्रीय घटनाओं 


Search again? (yes/no):  no


Program closed. Run again anytime to read news 👍


In [15]:
id="loop_ui_like"
print("===== AI NEWS READER =====")

name = input("Enter your name: ")
email = input("Enter your email: ")

while True:
    print("\nType your interest OR type 'exit' to quit")

    interest = input("Enter news topic: ")

    if interest.lower() == "exit":
        print("Program closed 👍")
        break

    language = input("Enter language (english/hindi): ").lower()

    print("\nFetching news...\n")

    if language == "hindi":
        url = f"https://newsapi.org/v2/everything?q={interest}&language=hi&apiKey={NEWS_API_KEY}"
    else:
        url = f"https://newsapi.org/v2/everything?q={interest}&language=en&apiKey={NEWS_API_KEY}"

    response = requests.get(url).json()
    articles = response.get("articles", [])[:5]

    for i, article in enumerate(articles):
        print(f"\n📰 News {i+1}: {article['title']}\n")

        summary = summarize_news(article)
        print(summary)

        print("-" * 50)

===== AI NEWS READER =====


Enter your name:  amit
Enter your email:  amit145@gmail.com



Type your interest OR type 'exit' to quit


Enter news topic:  business
Enter language (english/hindi):  english



Fetching news...


📰 News 1: Can Puck reinvent the news business for the influencer age?

**Simple Language:**

Puck is a new media company that makes newsletters for famous people. They give fans a chance to read exclusive stories and opinions from these celebrities. This way, people can stay up-to-date with the latest news and thoughts from their favorite stars.

**Key Points:**

- Puck is a media company that creates newsletters for famous people.
- These newsletters are part of a subscription bundle for fans.
- Famous people write exclusive content for their fans.

**Why it's important for exams:**

While this news may not seem directly related to exams, it highlights the changing way people consume news and information. In exams, you're often asked to analyze how media and technology are shaping society. Understanding how companies like Puck are adapting to the influencer age can help you think critically about the impact of media on our lives.

This news can also relate to exams

Enter news topic:  exit


Program closed 👍


In [ ]:
print("🚀 Welcome to ExamPulse AI")
print("Turning daily news into exam-winning insights\n")

name = input("Enter your name: ")
email = input("Enter your email: ")

while True:
    print("\n====== MAIN MENU ======")
    print("1. Get News")
    print("2. Exit")

    choice = input("Select an option (1/2): ")

    if choice == "2":
        print("Thank you for using ExamPulse AI 👋")
        break

    elif choice == "1":
        while True:
            print("\nType your interest OR type 'back' to return menu")

            interest = input("Enter news topic: ")

            if interest.lower() == "back":
                break

            language = input("Enter language (english/hindi): ").lower()

            print("\nExample exams: UPSC, SSC, Delhi Police, UP Police, BPSC, MBA, Banking")
            level = input("Enter your exam: ")

            print("\nFetching news...\n")

            # Language filter
            if language == "hindi":
                url = f"https://newsapi.org/v2/everything?q={interest}&language=hi&apiKey={NEWS_API_KEY}"
            else:
                url = f"https://newsapi.org/v2/everything?q={interest}&language=en&apiKey={NEWS_API_KEY}"

            response = requests.get(url).json()
            articles = response.get("articles", [])[:5]

            if not articles:
                print("No news found.")
                continue

            for i, article in enumerate(articles):
                print(f"\n📰 News {i+1}: {article.get('title', 'No title')}\n")

                text = article.get("title", "") + " " + str(article.get("description", ""))

                prompt = f"""
                You are an expert assistant helping students prepare for competitive exams.

                Explain the following news for a student preparing for: {level}

                Make it useful according to the exam type.

                Include:
                - Key points
                - Important facts
                - Why it is important for this exam
                - If possible, a potential exam question

                News:
                {text}
                """

                summary = llm.invoke(prompt).content
                print(summary)

                print("-" * 50)

    else:
        print("Invalid choice. Please select 1 or 2.")

🚀 Welcome to ExamPulse AI
Turning daily news into exam-winning insights

